In [11]:
import pandas as pd
import networkx as nx
import ast
from networkx.algorithms import bipartite
import matplotlib.pyplot as plt

Primero se genera el grafo bipartito con 2 conjuntos, personas y episodios. Un link entre una persona y episodio existe solo si esta trabajo en la produccion del capitulo.

In [12]:
# Cargar CSV
df = pd.read_csv("one_piece_merged.csv")

# Columnas donde hay listas de personas
staff_columns = ["guion", "arte", "animacion", "direccion"]

# Función para convertir string de lista a lista real
def parse_list(x):
    if pd.isna(x):
        return []
    try:
        return ast.literal_eval(x)
    except:
        return []

# Crear grafo bipartito
B = nx.Graph()

for _, row in df.iterrows():
    episodio = f"ep_{int(row['episodio'])}"
    
    # Agregar nodo episodio
    B.add_node(episodio, bipartite="episode")

    # Iterar por todas las áreas del staff
    for col in staff_columns:
        personas = parse_list(row[col])
        
        for persona in personas:
            persona_clean = persona.split("(")[0].strip()

            # Agregar nodo persona
            B.add_node(persona_clean, bipartite="staff")

            # Agregar arista
            B.add_edge(persona_clean, episodio)


print(f"Number of nodes: {B.number_of_nodes()}, Number of edges: {B.number_of_edges()}")
nx.write_gexf(B, "one_piece.gexf")

Number of nodes: 1386, Number of edges: 5183


Se procede a sacar la proyeccion del grafo, haciendo que solo queden las personas del staff.

In [13]:
from networkx.algorithms import bipartite

# Obtener nodos de staff
staff_nodes = {n for n, d in B.nodes(data=True) if d["bipartite"] == "staff"}

# Proyección
G_staff = bipartite.weighted_projected_graph(B, staff_nodes)

print(f"Number of nodes: {G_staff.number_of_nodes()}, Number of edges: {G_staff.number_of_edges()}")
nx.write_gexf(G_staff, "one_piece_staff_projection.gexf")

Number of nodes: 235, Number of edges: 2894


Centralidad de grado

In [14]:
degree = dict(G_staff.degree())

top = sorted(degree.items(), key=lambda x: x[1], reverse=True)[:10]

for name, deg in top:
    print(name, deg)

Kazuya Hisada 111
Jin Tanaka 108
Kenji Yokoyama 101
Miyuki Sato 86
Atsuhiro Tomioka 85
Masayuki Takagi 85
Miho Shiraishi 80
Eisaku Inoue 79
Shōji Yonemura 78
Masahiro Shimanuki 78


Centralidad de grado normalizada

In [ ]:
degree = nx.degree_centrality(G_staff)

top = sorted(degree.items(), key=lambda x: x[1], reverse=True)[:10]

for name, deg in top:
    print(name, deg)

Kazuya Hisada 0.4743589743589744
Jin Tanaka 0.46153846153846156
Kenji Yokoyama 0.43162393162393164
Miyuki Sato 0.36752136752136755
Atsuhiro Tomioka 0.3632478632478633
Masayuki Takagi 0.3632478632478633
Miho Shiraishi 0.3418803418803419
Eisaku Inoue 0.33760683760683763
Shōji Yonemura 0.33333333333333337
Masahiro Shimanuki 0.33333333333333337


Eigenvector centrality sin considerar peso de la arista

In [27]:
eigen = nx.eigenvector_centrality(G_staff)

top_eigen = sorted(eigen.items(), key=lambda x: x[1], reverse=True)[:10]

for name, val in top_eigen:
    print(name, val)

Kazuya Hisada 0.18577148470343574
Kenji Yokoyama 0.18172185177985042
Jin Tanaka 0.17385824082174936
Masahiro Shimanuki 0.15826170924453814
Toshio Deguchi 0.1540002297203062
Atsuhiro Tomioka 0.15184415229725662
Masayuki Takagi 0.1507705501762461
Miho Shiraishi 0.1458525022273851
Eisaku Inoue 0.14438529086653282
Tomohiro Nakayama 0.14407767928182671


Eigenvector centrality considerando el peso de la arista

In [26]:
eigen = nx.eigenvector_centrality(G_staff, weight='weight')

top_eigen = sorted(eigen.items(), key=lambda x: x[1], reverse=True)[:10]

for name, val in top_eigen:
    print(name, val)

Miyuki Sato 0.4042960986239657
Hirohiko Kamisaka 0.3083646613030738
Miho Shiraishi 0.30807389447136324
Ryūji Yoshiike 0.27243123651270973
Yoshiyuki Suga 0.24571504819739332
Kenji Yokoyama 0.21806691707464657
Masayuki Takagi 0.19880095964901093
Jin Tanaka 0.19679978687403013
Yoshihiro Ueda 0.1856788418575202
Michiyo Kawasaki 0.17646007689587193


Closeness

In [17]:
closeness = nx.closeness_centrality(G_staff)

top_closeness = sorted(closeness.items(), key=lambda x: x[1], reverse=True)[:10]

for name, val in top_closeness:
    print(name, val)

Kazuya Hisada 0.6554621848739496
Jin Tanaka 0.65
Kenji Yokoyama 0.6376021798365122
Masayuki Takagi 0.6109660574412533
Eisaku Inoue 0.6
Masahiro Shimanuki 0.6
Miho Shiraishi 0.5969387755102041
Atsuhiro Tomioka 0.5939086294416244
Toshio Deguchi 0.5909090909090909
Miyuki Sato 0.5909090909090909


In [18]:
betweenness = nx.betweenness_centrality(G_staff)

top_betweenness = sorted(betweenness.items(), key=lambda x: x[1], reverse=True)[:10]

for name, val in top_betweenness:
    print(name, val)

Jin Tanaka 0.09574135407378792
Kazuya Hisada 0.0939806321486594
Miyuki Sato 0.06781269177256295
Kenji Yokoyama 0.05643605769824752
Eisaku Inoue 0.03897772441513746
Masayuki Takagi 0.037571237184715216
Masahiro Kitazaki 0.03635322916514782
Atsuhiro Tomioka 0.034870000431142016
Shōji Yonemura 0.029331055399266236
Masahiro Shimanuki 0.029052835844405947
